# 🎬 Reels Autopilot — مجاني 100٪

**فيديو طويل ← أفضل اللحظات (Gemini) ← مقاطع 15 ثانية عمودية بترجمة متحركة ← نشر على YouTube / Instagram / Facebook / TikTok**

1. من القائمة: `Runtime → Change runtime type → T4 GPU` (مجاني — يسرّع الترجمة).
2. أضف مفاتيحك من أيقونة 🔑 **Secrets** على اليسار (فعّل *Notebook access*):
   `GEMINI_API_KEY` (إلزامي تقريباً) — ومفاتيح المنصات اختيارية (انظر README).
3. شغّل الخلايا بالترتيب ▶️

In [ ]:
# @title 1) التثبيت (≈ دقيقة)
!apt-get -qq install -y ffmpeg fonts-dejavu-core > /dev/null
!rm -rf /content/app && git clone -q --depth 1 -b claude/auto-reels-shorts-multiplatform-cy6p5z https://github.com/bluesky100090099/free-claude-code /content/app
%cd /content/app/reels_autopilot
!pip -q install -r requirements.txt

In [ ]:
# @title 2) تحميل المفاتيح من Colab Secrets
import os
from google.colab import userdata

KEYS = [
    "GEMINI_API_KEY",
    "YT_CLIENT_ID",
    "YT_CLIENT_SECRET",
    "YT_REFRESH_TOKEN",
    "IG_USER_ID",
    "IG_ACCESS_TOKEN",
    "FB_PAGE_ID",
    "FB_PAGE_TOKEN",
    "TIKTOK_CLIENT_KEY",
    "TIKTOK_CLIENT_SECRET",
    "TIKTOK_REFRESH_TOKEN",
]
for k in KEYS:
    try:
        os.environ[k] = userdata.get(k) or ""
    except Exception:
        pass
os.environ["WHISPER_DEVICE"] = "auto"  # يستخدم GPU تلقائياً إن وُجد
!python -m reels doctor

In [ ]:
# @title 3) اختر الفيديو والإعدادات
SOURCE = "upload"  # @param ["upload", "drive", "url"]
VIDEO_URL_OR_DRIVE_PATH = ""  # @param {type:"string"}
CLIPS = 5  # @param {type:"slider", min:1, max:20, step:1}
SECONDS = 15  # @param {type:"slider", min:5, max:60, step:1}
LANGUAGE = "ar"  # @param ["ar", "en", "fr", "es", "tr", "de"]
CROP = "smart"  # @param ["smart", "blur", "center"]
WHISPER_MODEL = "small"  # @param ["base", "small", "medium", "large-v3"]
os.environ["WHISPER_MODEL"] = WHISPER_MODEL

if SOURCE == "upload":
    from google.colab import files

    up = files.upload()
    INPUT = os.path.abspath(next(iter(up)))
elif SOURCE == "drive":
    from google.colab import drive

    drive.mount("/content/drive")
    INPUT = VIDEO_URL_OR_DRIVE_PATH  # e.g. /content/drive/MyDrive/videos/talk.mp4
else:
    INPUT = VIDEO_URL_OR_DRIVE_PATH
print("Input:", INPUT)

In [ ]:
# @title 4) ✂️ صناعة المقاطع
import subprocess, sys

proc = subprocess.run(
    [
        sys.executable,
        "-m",
        "reels",
        "make",
        INPUT,
        "-n",
        str(CLIPS),
        "-d",
        str(SECONDS),
        "--lang",
        LANGUAGE,
        "--crop",
        CROP,
        "-o",
        "/content/out",
    ],
    capture_output=True,
    text=True,
)
print(proc.stdout[-6000:], proc.stderr[-3000:])
BATCH = [
    l.split("=", 1)[1] for l in proc.stdout.splitlines() if l.startswith("BATCH_DIR=")
][-1]

In [ ]:
# @title 5) 👀 معاينة
import json, base64
from IPython.display import HTML, display

m = json.load(open(f"{BATCH}/manifest.json"))
html = "<div style='display:flex;flex-wrap:wrap;gap:16px'>"
for c in m["clips"]:
    b64 = base64.b64encode(open(f"{BATCH}/{c['file']}", "rb").read()).decode()
    html += (
        f"<div style='width:240px'><video width=240 controls src='data:video/mp4;base64,{b64}'></video>"
        f"<p dir='auto'><b>{c['title']}</b><br>{c['caption']}<br>{' '.join(c['hashtags'])}</p></div>"
    )
display(HTML(html + "</div>"))

In [ ]:
# @title 6) 🚀 النشر التلقائي (المنصات المُعدّة فقط)
PLATFORMS = "youtube,instagram,facebook,tiktok"  # @param {type:"string"}
NEXT = 0  # @param {type:"integer"}
# NEXT=0 ← انشر كل المقاطع الآن ، NEXT=1 ← مقطع واحد فقط (والباقي لاحقاً)
!python -m reels publish "$BATCH" --platforms "$PLATFORMS" --next $NEXT
!python -m reels status "$BATCH"

In [ ]:
# @title 7) 💾 تنزيل المقاطع + نصوص النشر (للنشر اليدوي)
import shutil
from google.colab import files

zip_path = shutil.make_archive("/content/reels", "zip", BATCH)
files.download(zip_path)
# أو احفظها في Drive:
# shutil.copytree(BATCH, "/content/drive/MyDrive/reels/" + os.path.basename(BATCH))